# 02 - Nettoyage et traitement des données

Ce notebook couvre la deuxième étape du projet : le nettoyage et la 
préparation du dataset pour l'analyse exploratoire et la modélisation.

**Objectifs :**
- Traiter les valeurs manquantes
- Détecter et traiter les doublons
- Détecter et traiter les valeurs aberrantes
- Sauvegarder le dataset nettoyé dans `data/processed/`

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# On se place à la racine du projet
os.chdir("..")

# Chargement du dataset brut
data_path = os.path.join("data", "raw", "Mental_Health_Lifestyle_Dataset.csv")
df = pd.read_csv(data_path, sep=",")

print(f"Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

Dataset chargé : 3000 lignes, 12 colonnes


In [3]:
# Nombre de doublons avant suppression
nb_doublons = df.duplicated().sum()
print(f"Nombre de doublons détectés : {nb_doublons}")

# Suppression des doublons
df = df.drop_duplicates()

print(f"Après suppression : {df.shape[0]} lignes restantes")

Nombre de doublons détectés : 0
Après suppression : 3000 lignes restantes


In [4]:
# Affichage des valeurs manquantes avant traitement
print("=== AVANT TRAITEMENT ===")
print(f"Nombre de lignes : {df.shape[0]}")
print(f"NaN dans Mental Health Condition : {df['Mental Health Condition'].isnull().sum()}")

# Suppression des lignes où Mental Health Condition est manquant
# On ne peut pas imputer la variable cible — ce serait inventer des diagnostics
df = df.dropna(subset=["Mental Health Condition"])

print("\n=== APRÈS TRAITEMENT ===")
print(f"Nombre de lignes : {df.shape[0]}")
print(f"NaN dans Mental Health Condition : {df['Mental Health Condition'].isnull().sum()}")

=== AVANT TRAITEMENT ===
Nombre de lignes : 3000
NaN dans Mental Health Condition : 595

=== APRÈS TRAITEMENT ===
Nombre de lignes : 2405
NaN dans Mental Health Condition : 0


In [6]:
# Colonnes numériques à vérifier
numerical_cols = ["Age", "Sleep Hours", "Work Hours per Week", 
                  "Screen Time per Day (Hours)", 
                  "Social Interaction Score", "Happiness Score"]

print("=== DÉTECTION DES VALEURS ABERRANTES (méthode IQR) ===\n")

outliers_summary = {}

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)   # Premier quartile
    Q3 = df[col].quantile(0.75)   # Troisième quartile
    IQR = Q3 - Q1                  # Écart interquartile
    
    # Bornes au-delà desquelles une valeur est aberrante
    borne_basse = Q1 - 1.5 * IQR
    borne_haute = Q3 + 1.5 * IQR
    
    # Détection des outliers
    outliers = df[(df[col] < borne_basse) | (df[col] > borne_haute)]
    outliers_summary[col] = len(outliers)
    
    print(f" {col}")
    print(f"   Q1={Q1:.2f} | Q3={Q3:.2f} | IQR={IQR:.2f}")
    print(f"   Bornes : [{borne_basse:.2f}, {borne_haute:.2f}]")
    print(f"   Outliers détectés : {len(outliers)}\n")

=== DÉTECTION DES VALEURS ABERRANTES (méthode IQR) ===

 Age
   Q1=30.00 | Q3=53.00 | IQR=23.00
   Bornes : [-4.50, 87.50]
   Outliers détectés : 0

 Sleep Hours
   Q1=5.50 | Q3=7.50 | IQR=2.00
   Bornes : [2.50, 10.50]
   Outliers détectés : 12

 Work Hours per Week
   Q1=30.00 | Q3=50.00 | IQR=20.00
   Bornes : [0.00, 80.00]
   Outliers détectés : 0

 Screen Time per Day (Hours)
   Q1=3.60 | Q3=6.60 | IQR=3.00
   Bornes : [-0.90, 11.10]
   Outliers détectés : 0

 Social Interaction Score
   Q1=3.30 | Q3=7.60 | IQR=4.30
   Bornes : [-3.15, 14.05]
   Outliers détectés : 0

 Happiness Score
   Q1=3.20 | Q3=7.50 | IQR=4.30
   Bornes : [-3.25, 13.95]
   Outliers détectés : 0



In [8]:
print("=== TRAITEMENT DES VALEURS ABERRANTES (écrêtage) ===\n")

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    borne_basse = Q1 - 1.5 * IQR
    borne_haute = Q3 + 1.5 * IQR
    
    # Écrêtage : on ramène les valeurs aux bornes
    df[col] = df[col].clip(lower=borne_basse, upper=borne_haute)
    
    print(f" {col} écrêtée entre {borne_basse:.2f} et {borne_haute:.2f}")

print(f"\nDataset après traitement : {df.shape[0]} lignes, {df.shape[1]} colonnes")

=== TRAITEMENT DES VALEURS ABERRANTES (écrêtage) ===

 Age écrêtée entre -4.50 et 87.50
 Sleep Hours écrêtée entre 2.50 et 10.50
 Work Hours per Week écrêtée entre 0.00 et 80.00
 Screen Time per Day (Hours) écrêtée entre -0.90 et 11.10
 Social Interaction Score écrêtée entre -3.15 et 14.05
 Happiness Score écrêtée entre -3.25 et 13.95

Dataset après traitement : 2405 lignes, 12 colonnes


In [9]:
print("=== VÉRIFICATION FINALE ===\n")

# Vérification des valeurs manquantes restantes
print("Valeurs manquantes restantes :")
print(df.isnull().sum())

# Vérification des doublons restants
print(f"\nDoublons restants : {df.duplicated().sum()}")

# Aperçu du dataset nettoyé
print(f"\nDimensions finales : {df.shape[0]} lignes, {df.shape[1]} colonnes")
display(df.head())

=== VÉRIFICATION FINALE ===

Valeurs manquantes restantes :
Country                        0
Age                            0
Gender                         0
Exercise Level                 0
Diet Type                      0
Sleep Hours                    0
Stress Level                   0
Mental Health Condition        0
Work Hours per Week            0
Screen Time per Day (Hours)    0
Social Interaction Score       0
Happiness Score                0
dtype: int64

Doublons restants : 0

Dimensions finales : 2405 lignes, 12 colonnes


,Country,Age,Gender,Exercise Level,Diet Type,Sleep Hours,Stress Level,Mental Health Condition,Work Hours per Week,Screen Time per Day (Hours),Social Interaction Score,Happiness Score
1,Australia,31,Male,Moderate,Vegan,4.9,Low,PTSD,48,5.2,8.2,6.8
3,Brazil,35,Male,Low,Vegan,7.2,Low,Depression,43,2.2,8.2,6.6
4,Germany,46,Male,Low,Balanced,7.3,Low,Anxiety,35,3.6,4.7,4.4
5,Japan,23,Other,Moderate,Balanced,2.7,Moderate,Anxiety,50,3.3,8.4,7.2
6,Japan,49,Male,Moderate,Junk Food,6.6,Low,Anxiety,28,7.2,5.6,6.9


In [11]:
# Chemin de sauvegarde dans le dossier processed
output_path = os.path.join("data", "processed", "mental_health_cleaned.csv")

# Sauvegarde sans l'index pandas (inutile dans le CSV)
df.to_csv(output_path, index=False)

print(f" Dataset nettoyé sauvegardé : {output_path}")
print(f"   {df.shape[0]} lignes | {df.shape[1]} colonnes")

 Dataset nettoyé sauvegardé : data\processed\mental_health_cleaned.csv
   2405 lignes | 12 colonnes


## Conclusions du nettoyage

- **Doublons** : aucun doublon
- **Valeurs manquantes** : 595 lignes supprimées (Mental Health Condition)
- **Outliers** : écrêtage appliqué sur les colonnes numériques
- **Dataset final** : 2405 lignes, 12 colonnes
- Fichier sauvegardé dans `data/processed/mental_health_cleaned.csv`
- Prochaine étape : analyse exploratoire (`03_eda.ipynb`)